In [1]:
import pandas as pd
import re

In [2]:
REGION = 'Nord'
enc = 'utf-8'

In [4]:
data = pd.read_csv(f'../../data/{REGION}/CM_{REGION}_Landcover_2001-2023_original.csv', encoding=enc)
data.columns = data.columns.str.lower()

In [5]:
data.head()

,x,y,2001_01_01_lc_prop1,2001_01_01_lc_prop1_assessment,2001_01_01_lc_prop2,2001_01_01_lc_prop2_assessment,2001_01_01_lc_prop3,2001_01_01_lc_prop3_assessment,2001_01_01_lc_type1,2001_01_01_lc_type2,...,2023_01_01_lc_prop2_assessment,2023_01_01_lc_prop3,2023_01_01_lc_prop3_assessment,2023_01_01_lc_type1,2023_01_01_lc_type2,2023_01_01_lc_type3,2023_01_01_lc_type4,2023_01_01_lc_type5,2023_01_01_lw,2023_01_01_qc
0,13.56021,9.29772,31,94,36,93.0,30,94,12,12,...,98.0,30,99,12,12,1,6,7,2,0
1,13.59642,9.22586,31,92,36,91.0,30,92,12,12,...,99.0,30,99,12,12,3,5,8,2,0
2,13.59642,9.26179,31,97,36,96.0,30,97,12,12,...,97.0,30,98,12,12,1,6,7,2,0
3,13.59642,9.29772,31,97,36,96.0,30,97,12,12,...,97.0,30,98,12,12,1,6,7,2,0
4,13.63262,9.18992,31,92,36,92.0,30,92,12,12,...,99.0,30,99,12,12,1,6,7,2,0


In [6]:
lc_columns = data.columns.tolist()
lc_columns = [x.split('_')[0] for x in lc_columns]

lc_columns
lc_years = set()

for i in range(0, len(lc_columns)):
    try:
        lc_years.add(int(lc_columns[i]))
    except ValueError:
        pass

lc_years = list(lc_years)

In [7]:
dates_ = re.compile(r"[0-9]{4}_[0-9]{2}_[0-9]{2}_")
landcover_datasets_per_year = []

for year in lc_years:
    frame_name = f'landcover_{year}'
    location_columns = ['x', 'y']
    filtered_columns = [col for col in data if col.startswith(f'{year}')]
    columns_to_keep = location_columns + filtered_columns
    locals()[frame_name] = data[columns_to_keep].copy()
    locals()[frame_name].insert(2, 'year', int(year))
    locals()[frame_name] = locals()[frame_name].rename(columns=lambda x: re.sub(dates_,'',x))
    landcover_datasets_per_year.append(locals()[frame_name])

In [8]:
landcover_2024 = landcover_2023.copy()
landcover_2024['year'] = 2024
landcover_datasets_per_year.append(landcover_2024)

In [9]:
landcover_processed = pd.concat(landcover_datasets_per_year, ignore_index=True)
landcover_processed.reset_index(drop=True, inplace=True)

In [10]:
landcover_processed

,x,y,year,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc
0,13.56021,9.29772,2001,31,94,36,93.0,30,94,12,12,1,6,7,2,0
1,13.59642,9.22586,2001,31,92,36,91.0,30,92,12,12,1,6,7,2,0
2,13.59642,9.26179,2001,31,97,36,96.0,30,97,12,12,3,5,8,2,0
3,13.59642,9.29772,2001,31,97,36,96.0,30,97,12,12,3,5,8,2,0
4,13.63262,9.18992,2001,31,92,36,92.0,30,92,12,12,3,5,8,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95011,15.33423,8.11192,2024,31,99,36,95.0,30,99,12,12,3,5,8,2,0
95012,15.33423,8.14786,2024,22,93,20,93.0,20,93,9,9,4,4,4,2,0
95013,15.37044,8.00412,2024,22,93,20,93.0,20,93,9,9,4,4,4,2,0
95014,15.37044,8.04005,2024,22,91,20,91.0,20,91,9,9,4,4,4,2,0


In [11]:
landcover_processed.to_csv(f'../../data/{REGION}/CM_{REGION}_Landcover_2001-2024_processed.csv', index=False, encoding=enc)